# 03 – PostgreSQL database

**Goal:** Store the integrated tables in PostgreSQL (container in the Codespace) and query them from Python with SQL, including joins and aggregations.
**Input:** `data/interim/*.parquet` (notebook 02) · **Output:** filled database tables
**Assessment items:** P3 (database with SQL queries from Python), B2 (PostgreSQL instead of SQLite, joins and aggregations)

Schema: `sql/schema.sql` – primary and foreign keys connect `circuits` → `races` → `results` / `laps` / `weather` / `pairs`, and `drivers` → `driver_map` / `results` / `laps` / `pairs`.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
from sqlalchemy import text

from src.config import INTERIM_DIR
from src.db import TABLE_ORDER, get_engine, load_tables, reset_schema, table_counts

engine = get_engine()
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one().split(",")[0])

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu


## 1. Create schema and load the tables

`reset_schema` drops and recreates all project tables, so the notebook always produces the same database state.

In [3]:
reset_schema(engine)
tables = {name: pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
          for name in TABLE_ORDER if name != "pairs"}
load_tables(engine, tables)
pd.Series(table_counts(engine)).to_frame("rows")

,rows
circuits,33
driver_map,197
drivers,45
laps,211297
pairs,0
races,192
results,3876
weather,28923


## 2. SQL queries from Python

**Query 1 – races and laps per era and circuit type** (join `races` × `circuits`, aggregation)

In [4]:
pd.read_sql("""
    SELECT r.era,
           c.circuit_type,
           COUNT(*)                 AS races,
           ROUND(AVG(r.total_laps)) AS avg_laps,
           ROUND(AVG(c.length_km), 3) AS avg_length_km
    FROM races r
    JOIN circuits c USING (circuit_id)
    GROUP BY r.era, c.circuit_type
    ORDER BY r.era, races DESC
""", engine)

,era,circuit_type,races,avg_laps,avg_length_km
0,E1,race,68,60.0,5.231
1,E1,street,13,63.0,4.873
2,E2,race,61,61.0,5.171
3,E2,street,31,60.0,5.181
4,E3,race,12,60.0,5.243
5,E3,street,7,62.0,4.964


**Query 2 – pit stops and neutralised laps per race** (join `laps` × `races` × `circuits`, conditional aggregation)

In [5]:
per_race = pd.read_sql("""
    SELECT r.season, r.round, r.race_name, c.circuit_type,
           COUNT(*)                                       AS laps,
           COUNT(*) FILTER (WHERE l.pit_in)               AS pit_stops,
           ROUND(100.0 * AVG((l.track_status ~ '[4567]')::int), 1) AS pct_laps_sc_vsc_red
    FROM laps l
    JOIN races r    USING (race_id)
    JOIN circuits c USING (circuit_id)
    GROUP BY r.season, r.round, r.race_name, c.circuit_type
    ORDER BY r.season, r.round
""", engine)
per_race.head(10)

,season,round,race_name,circuit_type,laps,pit_stops,pct_laps_sc_vsc_red
0,2018,1,Australian Grand Prix,street,940,22,10.6
1,2018,2,Bahrain Grand Prix,race,998,34,5.8
2,2018,3,Chinese Grand Prix,race,1115,31,9.2
3,2018,4,Azerbaijan Grand Prix,street,848,36,25.5
4,2018,5,Spanish Grand Prix,race,1021,23,14.1
5,2018,6,Monaco Grand Prix,street,1516,27,2.8
6,2018,7,Canadian Grand Prix,street,1219,21,6.1
7,2018,8,French Grand Prix,race,920,22,13.6
8,2018,9,Austrian Grand Prix,race,1244,24,3.6
9,2018,10,British Grand Prix,race,902,29,17.6


In [6]:
per_race.groupby("season")[["pit_stops", "pct_laps_sc_vsc_red"]].mean().round(1)

,pit_stops,pct_laps_sc_vsc_red
season,,
2018,27.0,8.7
2019,32.3,6.6
2020,36.4,10.8
2021,38.5,8.1
2022,38.0,10.2
2023,42.9,6.2
2024,35.4,5.1
2025,35.0,8.4
2026,42.9,10.3


**Query 3 – positions gained from grid to finish by era and starting group** (join `results` × `races`, CASE, aggregation)

In [7]:
pd.read_sql("""
    SELECT r.era,
           CASE WHEN res.grid BETWEEN 1 AND 5  THEN '1: P1-5'
                WHEN res.grid BETWEEN 6 AND 10 THEN '2: P6-10'
                WHEN res.grid BETWEEN 11 AND 15 THEN '3: P11-15'
                ELSE '4: P16+ / pit lane' END        AS start_group,
           COUNT(*)                                 AS starts,
           ROUND(AVG(res.grid - res.finish_pos), 2) AS avg_positions_gained
    FROM results res
    JOIN races r USING (race_id)
    WHERE res.finish_pos IS NOT NULL AND res.grid > 0
    GROUP BY r.era, start_group
    ORDER BY start_group, r.era
""", engine)

,era,start_group,starts,avg_positions_gained
0,E1,1: P1-5,405,-2.30
1,E2,1: P1-5,460,-2.05
2,E3,1: P1-5,95,-2.65
3,E1,2: P6-10,404,-1.60
4,E2,2: P6-10,458,-1.67
5,E3,2: P6-10,95,-1.51
6,E1,3: P11-15,401,0.42
7,E2,3: P11-15,459,0.27
8,E3,3: P11-15,95,0.65
9,E1,4: P16+ / pit lane,380,3.39
